# 07 Post Content Clustering
Cluster posts via K-Means, GMM, and optional HDBSCAN.

In [1]:
import warnings
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "utils" / "utils.py").exists():
    if (PROJECT_ROOT / "notebooks" / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT / "notebooks"
    elif (PROJECT_ROOT.parent / "utils" / "utils.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from utils.utils import ensure_project_dirs, load_raw_dataset, clean_dataset, PROCESSED_DIR, REPORTS_DIR, FIGURES_DIR
from utils.features import engineer_kpis, build_post_feature_sets, aggregate_business_features
from utils.evaluation import regression_metrics, rank_models
from utils.visualization import set_plot_style, save_figure

set_plot_style()
ensure_project_dirs()
RAW_DATA_PATH = Path("../jsons/all_final_appended.json")
if not RAW_DATA_PATH.exists():
    RAW_DATA_PATH = PROJECT_ROOT / "synthetic_generator" / "synthetic_social_media_posts.csv"
KPI_PATH = PROCESSED_DIR / "kpi_dataset.csv"

## Load KPI and Feature Matrix

In [2]:
if KPI_PATH.exists():
    df = pd.read_csv(KPI_PATH, parse_dates=["post_date"])
else:
    df = engineer_kpis(clean_dataset(load_raw_dataset(RAW_DATA_PATH)))
df.head()

,business_name,sector,followers_count,post_date,posting_hour,day_of_week,month,post_type,caption_text,caption_length,...,is_reel,posting_time_bin,caption_length_bin,hashtags_count_bin,emoji_count_bin,engagement_level,business_size_bin,high_engagement,high_view_rate,high_comment_rate
0,LOFT Palestine,Fashion,4392.0,2026-03-31,0.0,Tuesday,3,reel,إطلالة أنثوية بلمسة عصرية ✨ من LOFT. للطلب الت...,88,...,1,night,medium,none,few,medium,small,0,0,0
1,LOFT Palestine,Fashion,4392.0,2026-03-18,0.0,Wednesday,3,reel,اختيارك المثالي لإطلالة كاجول مميزة 🤍✨ LOFT كو...,111,...,1,night,medium,none,few,high,small,1,1,0
2,LOFT Palestine,Fashion,4392.0,2026-03-16,0.0,Monday,3,reel,ستايل شبابي بلمسة عصرية له ولها ✨ اختيارات ممي...,133,...,1,night,medium,none,few,high,small,1,1,1
3,LOFT Palestine,Fashion,4392.0,2026-03-07,0.0,Saturday,3,reel,ستايل يناسبك، وتجربة تستحق الزيارة ✨ زورونا في...,82,...,1,night,medium,none,few,high,small,1,1,0
4,LOFT Palestine,Fashion,4392.0,2026-03-05,0.0,Thursday,3,reel,#LOFTSTYLE #Menstyle,21,...,1,night,short,few,none,high,small,1,1,0


In [3]:
import importlib.util
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
has_hdbscan = importlib.util.find_spec("hdbscan") is not None
if has_hdbscan:
    import hdbscan

feat = ["engagement_rate","view_rate","comment_rate","like_rate","view_engagement_rate","caption_length","hashtags_count","emoji_count","discount_percent","posting_hour"]
X = StandardScaler().fit_transform(df[feat].fillna(0))

## Experiments and Metric Comparison

In [4]:
rows, labels_store = [], {}
for k in range(3,11):
    y = KMeans(n_clusters=k, random_state=42, n_init=20).fit_predict(X)
    rows.append({"model":"kmeans","setting":f"k={k}","silhouette":silhouette_score(X,y),"davies_bouldin":davies_bouldin_score(X,y),"calinski_harabasz":calinski_harabasz_score(X,y),"clusters":len(np.unique(y)),"noise_ratio":0.0})
    labels_store[("kmeans",f"k={k}")] = y

for k in range(3,11):
    for cov in ["full","tied","diag"]:
        y = GaussianMixture(n_components=k, covariance_type=cov, random_state=42).fit_predict(X)
        if len(np.unique(y)) < 2:
            continue
        rows.append({"model":"gmm","setting":f"components={k},cov={cov}","silhouette":silhouette_score(X,y),"davies_bouldin":davies_bouldin_score(X,y),"calinski_harabasz":calinski_harabasz_score(X,y),"clusters":len(np.unique(y)),"noise_ratio":0.0})
        labels_store[("gmm",f"components={k},cov={cov}")] = y

if has_hdbscan:
    for mcs in [5,10,20]:
        y = hdbscan.HDBSCAN(min_cluster_size=mcs).fit_predict(X)
        valid = y >= 0
        if len(np.unique(y[valid])) < 2:
            continue
        rows.append({"model":"hdbscan","setting":f"min_cluster_size={mcs}","silhouette":silhouette_score(X[valid],y[valid]),"davies_bouldin":davies_bouldin_score(X[valid],y[valid]),"calinski_harabasz":calinski_harabasz_score(X[valid],y[valid]),"clusters":len(np.unique(y[valid])),"noise_ratio":float((y==-1).mean())})
        labels_store[("hdbscan",f"min_cluster_size={mcs}")] = y

exp = rank_models(pd.DataFrame(rows), higher_is_better_cols=["silhouette","calinski_harabasz","clusters"], lower_is_better_cols=["davies_bouldin","noise_ratio"])
best = exp.iloc[0]
y = labels_store[(best["model"], best["setting"])]
out = df.copy()
out["cluster_id"] = y
profiles = out[out["cluster_id"] >= 0].groupby("cluster_id", as_index=False).agg(
    posts=("cluster_id","size"),
    engagement_rate=("engagement_rate","mean"),
    view_rate=("view_rate","mean"),
    comment_rate=("comment_rate","mean"),
    top_post_type=("post_type", lambda x: x.mode().iat[0]),
    top_time=("posting_time_bin", lambda x: x.mode().iat[0]),
)
labels = ["high-engagement storytellers","reach-focused visuals","discussion starters","promotion-heavy posts","steady baseline content","niche audience posts","community updates","experimental formats"]
profiles = profiles.sort_values("engagement_rate", ascending=False).reset_index(drop=True)
profiles["cluster_label"] = [labels[i] if i < len(labels) else f"cluster_{i}" for i in range(len(profiles))]
out = out.merge(profiles[["cluster_id","cluster_label"]], on="cluster_id", how="left")

## Save Outputs

In [5]:
out.to_csv(PROCESSED_DIR / "post_clusters.csv", index=False)
profiles.to_csv(PROCESSED_DIR / "cluster_profiles.csv", index=False)
exp.to_csv(REPORTS_DIR / "post_clustering_experiments.csv", index=False)
display(exp.head(15))
display(profiles)
print("Insight: cluster archetypes support reusable content playbooks.")

,model,setting,silhouette,davies_bouldin,calinski_harabasz,clusters,noise_ratio,composite_score
0,kmeans,k=10,0.349829,0.810373,229.421365,10,0.0,4.461786
1,gmm,"components=10,cov=tied",0.479077,0.636705,182.095018,10,0.0,4.456914
2,kmeans,k=9,0.404636,0.730815,213.142007,9,0.0,4.338212
3,kmeans,k=8,0.462300,0.775242,204.910400,8,0.0,4.213774
4,gmm,"components=9,cov=tied",0.473625,0.794710,167.410864,9,0.0,4.174641
5,gmm,"components=8,cov=tied",0.652435,0.861831,147.480147,8,0.0,4.143090
6,gmm,"components=7,cov=tied",0.647844,0.599794,142.200067,7,0.0,4.063358
7,kmeans,k=7,0.463496,0.805372,190.242585,7,0.0,3.987207
8,gmm,"components=6,cov=tied",0.643546,0.897720,134.411318,6,0.0,3.766297
9,kmeans,k=6,0.471136,0.903862,162.942944,6,0.0,3.680189


,cluster_id,posts,engagement_rate,view_rate,comment_rate,top_post_type,top_time,cluster_label
0,4,4,1.025085,41.352667,0.003916,reel,night,high-engagement storytellers
1,9,2,0.645851,65.580017,0.032176,reel,night,reach-focused visuals
2,2,8,0.402784,18.210203,0.006298,reel,night,discussion starters
3,6,3,0.053707,0.077321,0.000224,Reel,night,promotion-heavy posts
4,7,7,0.011304,312.545287,0.000220,Reel,night,steady baseline content
5,0,14,0.010296,4.645440,0.000065,reel,night,niche audience posts
6,1,353,0.008162,1.102307,0.000266,reel,night,community updates
7,8,216,0.004574,0.879150,0.000235,reel,night,experimental formats
8,3,31,0.003216,0.261238,0.000491,reel,night,cluster_8
9,5,18,0.001069,0.095765,0.000021,reel,night,cluster_9


Insight: cluster archetypes support reusable content playbooks.
